# CheMLFlow Tutorial 00: Dataset EDA

This notebook shows the simplest useful CheMLFlow workflow:

- load a local CSV
- run the generic EDA node
- inspect the generated plots

For this example, we use the bundled PGP dataset.

## 1. Choose the repo to clone

The default points at your `origin` fork. Override `CHEMLFLOW_TUTORIAL_REPO` if you want a different URL.

In [ ]:
import os

REPO_URL = os.environ.get("CHEMLFLOW_TUTORIAL_REPO", "https://github.com/bsmith24/CheMLFlow.git")
print(REPO_URL)

## 2. Clone the repo and move into it

In [ ]:
%cd /content
!rm -rf CheMLFlow
!git clone "$REPO_URL"
%cd /content/CheMLFlow

## 3. Install the minimum dependencies for this tutorial

This tutorial only needs the packages required to import `main.py`, load the CSV, and generate the EDA plots.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install PyYAML pandas numpy scipy scikit-learn matplotlib seaborn joblib xgboost

## 4. Write the config

The config is inline here on purpose so you can see exactly what CheMLFlow is running.

In [ ]:
from pathlib import Path
from textwrap import dedent

CONFIG_PATH = Path("tutorials/00_dataset_eda/configs/pgp_raw_eda_colab.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config_yaml = dedent(
    """
    global:
      pipeline_type: pgp
      task_type: classification
      base_dir: tutorials/00_dataset_eda/artifacts/data
      run_dir: tutorials/00_dataset_eda/artifacts/run
      target_column: Activity
      random_state: 42
      thresholds:
        active: 1000
        inactive: 10000

    pipeline:
      nodes:
        - get_data
        - analyze.eda

    get_data:
      data_source: local_csv
      source:
        path: local_data/pgp_broccatelli.csv

    analyze:
      eda:
        include:
          overview: true
          missingness: true
          numeric_histograms: true
          correlation_heatmap: true
          target_distribution: true
          class_balance: true
          descriptor_scatter: false
          descriptor_boxplots: false
    """
).strip() + "\n"

CONFIG_PATH.write_text(config_yaml, encoding="utf-8")
print(config_yaml)

## 5. Launch CheMLFlow

In [ ]:
import os

os.environ["CHEMLFLOW_CONFIG"] = str(CONFIG_PATH)
os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = "/content/.mplconfig"

!python main.py

## 6. Inspect the manifest and generated files

In [ ]:
from pathlib import Path
import json

eda_dir = Path("tutorials/00_dataset_eda/artifacts/run/eda")
manifest = json.loads((eda_dir / "eda_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
print("\nGenerated files:")
for path in sorted(eda_dir.iterdir()):
    print(path.name)

## 7. Display the plots

In [ ]:
from IPython.display import Image, display

for name in [
    "dataset_overview.png",
    "missingness.png",
    "numeric_histograms.png",
    "target_distribution.png",
    "class_balance.png",
]:
    path = eda_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))